In [11]:
import torch
import torch.nn as nn

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

DATA PREPARATION!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# dataset
df = pd.read_csv("fmnist_small.csv")

# split train and test
X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

# Scale only the pixels not the labels!
X_train_scaled = X_train/255.0
X_test_scaled = X_test/255.0

# Convert all to tensors
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)

'\nX_train_tensor = torch.from_numpy(X_train_scaled.values).float()\nX_test_tensor = torch.from_numpy(X_test_scaled.values).float()\ny_train_tensor = torch.from_numpy(y_train.values).long()\ny_test_tensor = torch.from_numpy(y_test.values).long()\n'

DATA LOADING!

In [13]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train_tensor, y_train_tensor):
        self.X_train_tensor = X_train_tensor
        self.y_train_tensor = y_train_tensor
    def __len__(self):
        return len(self.X_train_tensor)
    def __getitem__(self,idx):
        return self.X_train_tensor[idx], self.y_train_tensor[idx]

train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory=True)

MODEL ARCHITECTURE!

In [14]:
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64, 10)
    )
  def forward(self, x):
    return self.model(x)

In [ ]:
from torch import optim

# set learning rate and epochs
epochs = 10
learning_rate = 0.1
lambda_val = 1e-4

# instatiate the model
model = MyNN(X_train.shape[1]).to(device)
# loss function
loss_fun = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=lambda_val)
'''
Note: There is no param for l1, we need to manually add it while training! like this:
lambda_l1 = 0.001
outputs = model(batch_features)
loss = loss_fun(outputs, batch_labels)
l1_penalty = 0
for param in model.parameters():
    l1_penalty += torch.sum(torch.abs(param))
loss = loss + lambda_l1 * l1_penalty
'''

'\nNote: There is no param for l1, we need to manually add it while training! like this:\nlambda_l1 = 0.001\noutputs = model(batch_features)\nloss = loss_fun(outputs, batch_labels)\nl1_penalty = 0\nfor param in model.parameters():\n    l1_penalty += torch.sum(torch.abs(param))\nloss = loss + lambda_l1 * l1_penalty\n'

TRAINING LOOP!

In [ ]:
# training loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    # zero the prev grads
    optimizer.zero_grad()
    # move the tensor data to GPU!
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    # forward pass
    outputs = model(batch_features)
    # calculate loss
    loss = loss_fun(outputs, batch_labels)
    # back pass
    loss.backward()
    # update grads
    optimizer.step()
    # calculate loss for all items in a batch
    total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 0.9815057134628296
Epoch: 2 , Loss: 0.6846812399228414
Epoch: 3 , Loss: 0.6111650077501932
Epoch: 4 , Loss: 0.5620819628238678
Epoch: 5 , Loss: 0.535804118613402
Epoch: 6 , Loss: 0.508719345331192
Epoch: 7 , Loss: 0.4899301420648893
Epoch: 8 , Loss: 0.45951394230127335
Epoch: 9 , Loss: 0.42842212279637654
Epoch: 10 , Loss: 0.42754427552223206


EVALUATION LOOP ON TEST DATA!

In [17]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_test_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8333333333333334


EVALUATION LOOP ON TRAIN DATA!

In [18]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_train_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.886875
